<a href="https://colab.research.google.com/github/dipikamishra/my-repository/blob/main/04_AssociationPatternMining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Association Pattern Mining

In this week's practical we will learn and practice the following skills:
- Explore how transaction data are stored
- Use apriori and FP growth algorithms to find frequent item sets
- Explore the relationship between itemset size and frequency
- Generate association rules
- Understand support, confidence, and lift
- Interpret association rules, hypothesize relationships, and propose further work.
- Documenting our work

Apriori builds combinations gradually

* Apriori generates candidates;
* FP-Growth compresses transactions into a tree.

tiny example:

Transaction	Items bought

* T1	Bread, Milk
* T2	Bread, Diapers, Beer, Eggs
* T3	Milk, Diapers, Beer, Cola
* T4	Bread, Milk, Diapers, Beer
* T5	Bread, Milk, Diapers, Cola


## Preparing your workspace
In this practical we'll be using Python/[Pandas](https://pandas.pydata.org) to explore and visualise some example data using [Jupyter Notebooks](https://jupyter.org) in [Google Colaboratory](https://colab.research.google.com).

At the start of each practical you should make a copy of the notebook in your own google drive:
- File -> Save as copy in Drive

If you would prefer to work with a Jupyter Lab session ony your own machine you can download the notebook directly via:
- File -> Downaload -> Download .ipynb

In this practial we will be preparing data for use with some data mining applications.

The data for this week can be found on [GitHub](https://github.com/PaulHancock/COMP5009_pracs).


## Animals of NSW

Our task for today is to use association pattern mining to understand the co-occorance of animals across NSW.
There are many reasons that animals will be observed in the same regions including:
- They feed on similar plants or animals,
- They prefer the same habitat for breeding, mating, or resting,
- The animals have a relationship with each other such as predator/prey,
- The animals are ubiquitous in NSW and thus can be expected to be found everywhere you look, and their co-occurance is expected by chance alone.

Before we can start to explore the reasons for co-occurance we need to first identify which animals are observed together, and thus we will use assocation pattern mining to find these patterns.

Once we have established patterns of occurance we can generate association rules, which may give some insight as to the reason by the co-occurances.

### The data
The data set that we will be using this week is taken from the [NSW BioNet Atlas](https://doi.org/10.15468/14jd9g).
The data contain records from the NSW Department of planning, industry and environment BioNet Atlas database of flora and fauna sightings.
It includes records from other custodians such as the National Herbarium of NSW, Forests NSW, Australian Bird and Bat Banding Scheme and the Australian Museum.
The full data set is 24GB of detailed sighting information, with 14,861,875 observations (rows) and 183 features (columns).
Many of the features are either empty, not relevant, or have been redacted.

For this work I have transformed the data as follows:
- Select only rows which contain animals ("kingdom" == "animalia")
- Select only rows which contains observations "year"==2010
- Select features "vernacularName" (a.k.a "Name") and "county" (a.k.a "Region")
- The above selection produces 7,441,321 instaces and two features ("Name", "Region")
- There are 2475 unique animal names, and 173 unique regions. A random sample of 10 of the top 800 most common animals are selected, and instances that don't contain these animals are removed.
- The data are then transformed by grouping by "Region" to obtain a list of the unique animals names that were observed.
- Finally, the data are one-hot encoded into a table of regions and features to form 157 rows x 10 features.

The animals that are present in the data set are:
- Australasian Shoveler
- Mountain Brushtail Possum
- Red-rumped Parrot
- Barrabarruun
- Watson's Tree Frog
- Mainland Black-faced Cuckoo-shrike
- Eastern Bent-winged Bat
- Common Planigale
- Black-faced Monarch
- Whiptail Wallaby

A `.csv` of these "transactions" has been made available on [GitHub](https://github.com/PaulHancock/COMP5009_pracs/raw/refs/heads/main/data/animals_transactions_2010.csv)


**References / further reading:**
- NSW Department of planning, industry and environment (2023). NSW BioNet Atlas. Occurrence dataset https://doi.org/10.15468/14jd9g accessed via GBIF.org on 2025-07-17.




In [ ]:
import pandas as pd
import urllib
import urllib.request
import numpy as np

## Load the data file into a pandas data frame

Pands will let us read a file directly from a url.

Once loaded you should determine the data size and dimension, as well as the features that are present, and the data types.

In [ ]:
# Load data
data_url = '?'
df = pd.read_csv(data_url)
df

In [ ]:
# Load data
data_url = 'https://github.com/PaulHancock/COMP5009_pracs/raw/refs/heads/main/data/animals_transactions_2010.csv'
df = pd.read_csv(data_url)
df

In [ ]:
# determine the size and dimension of the data
? df.shape.

In [ ]:
# Look for missing data
?

In [ ]:
# fix missing data
?

In [ ]:
# identify data types
?

Confirm that the data shape and type is as you exepct, and that there are no missing data.
If there are any issues, you should address them.

## Use the Apriori algorithm to find frequent itemsets

Select the Apriori algorithm and perform frequent itemset mining with `minsup = 0.2`.

Determine the number of frequent 2-itemsets, and 3-itemsets.
- The best three rules with largest confidence. Examine these rules and describe them in your own words.

The `apriori` algorithm is found in the `mlxtend` package, so we import it along with the `association_rules` function.

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

In [ ]:
ap_itemsets = apriori(df,
                      min_support=?,  # choose the (relative) minsup
                      use_colnames=True)

In [ ]:
ap_itemsets

Now that we have our itemsets we want to chose those with `2<=k<=3`.
This isn't explicitly stored within our dataframe so we'll make a new column which is just the value of `len(itemsets)`.

In [ ]:
def find_k(row):
  """Return the number of items in the itemset"""
  return len(row['itemsets'])

# Create a new column which counts the number of items in the itemset
ap_itemsets['k'] = ap_itemsets.apply(?, # Apply the function `find_k`
                                     axis=1) # apply the function to each row

In [ ]:
ap_itemsets

Us `groupby` and `count` to determine the number of item sets for each size of item set.

In [ ]:
ap_itemsets.?

Now use `nlargest` to list the 10 itemsets with the highest support

In [ ]:
# Now lets see the top 10 itemsets
# try either .head() or .nlargest(10,'support')
ap_itemsets

Are the top 10 itemsets are all 1-itemsets? Is this surprising to you?

If there are any 2 or 3 itemsets in the top 10, identify the animals and see if you can see any comonality between them, in terms of habitat requirements, diet, range, etc.
Make an initial hypothesis about what is driving the co-occurance of these animals.

## Turn item-sets into association rules

Now that we have identified frequent item-sets we can start to look at the links between them using association rules.

We use these itemsets to generate association rules with a minimum confidence of 0.8.

In [ ]:
ap_rules = association_rules(ap_itemsets,
                             metric=?,
                             min_threshold=?) # choose the minimum confidence value

In [ ]:
ap_rules.head()

The above table shows a range of different measures for the various rules. Of particular interest to us are the following columns:
- Antecedences and Consequents as they are the A,C for Conf(A->C)
- The support of the antecedent and consequent
- Suppot for the item set of A union C
- Confidence and lift of the association rule

Note that the rules above are not sorted by confidence. We should do that ourselves by using the `sort_values` function.

In [ ]:
ap_rules.sort_values(?, ascending=False)

Describe the first three that you see above in your own words.

The [Eastern Bent-winged Bat](https://www.nationalparks.nsw.gov.au/plants-and-animals/eastern-bentwing-bat) is fun little fellow, but hard to find, due to their small size and nocturnal nature.
If we were to go looking for this bat, what other animals could we use as a proxy to tell that we were looking in the right place?

That is - if B is the Eastern Bent-winged Bat, what animal(s) A would give a rule Conf(A -> B) with high confidence and lift?

In [ ]:
# choose all the rules wihch have our bat as the consquent
bat_rules = ap_rules[ap_rules.consequents == frozenset([?])]

# choose the three rules with the highest confidence
bat_rules.sort_values(?,
                        ascending=False).head(3) # choose the top 3 only

## Use the FP-Growth algorithm

Use the FP-Growth algorithm to generate item sets and compare them to those found by the Apriori algorithm.

What are the main differences that you see between the results of the two algorithms?

In [ ]:
from mlxtend.frequent_patterns import fpgrowth

In [ ]:
fp_itemsets = fpgrowth(df,
                       min_support=?, # Same as before
                       use_colnames=True)

# add our label for the size of the itemset
fp_itemsets['k'] = fp_itemsets.apply(?, axis=1)

fp_itemsets.head(10)

In [ ]:
ap_itemsets.head(10)

## Wrap up

Summarise your work for today.
Include a description of:
- any data cleaning activities that you engaged in,
- the task that was to be completed,
- any differences that you saw between the two algorithms that we used.

Consider now that the data set contained all 2475 animals over 173 regions and all years of study.
What are some of the issues that you might run into when working with this data set?
Consider issues related to the practical processinng of data, as well as issues related to the interpretation of results.
